#### mysql과 python 연동에서의 유의할 점 
- 서버의 정보(주소, port, 아이디, 비밀번호, 데이터베이스명)를 코드에 그대로 입력 한 뒤 github에 코드를 업로드 
    - 보안상 취약점 
        - 내 집 주소와 현관문의 비밀번호를 알려주는 꼴
        - github에 해당 데이터들을 업로드를 하면 -> 확인하고 파일에 대해서 락 
        - openkey -> openapi와 같은 곳에서 비밀번호를 제공 
        - 이런 민감한 정보들은 숨겨서 관리 -> python-dotenv 라이브러리 설치하여 사용

In [26]:
# !pip install python-dotenv

In [1]:
# 설치 할때 사용하는 라이브러리의 이름과 import 할때의 이름이 다른 경우들이 종종 발생 
from dotenv import load_dotenv
import pymysql 
import os 
import pandas as pd

In [2]:
# os -> 환경변수 (python 환경에서 사용하는 변수들에 접근하기 위해 사용)
# load_dotenv -> 환경변수에 등록 (.env 파일의 내용을 환경 변수에 등록(임시 등록))
load_dotenv()

True

In [29]:
# 등록된 환경변수에 접근 
# 가져온다(get) + 환경(env) -> getenv(변수명) 
os.getenv('pwd')

'1234'

In [30]:
# DB와 연결 
_db = pymysql.connect(
    host = os.getenv('host'),  
    # getenv() 함수의 결과는 문자 -> port는 숫자를 사용 -> 문자를 숫자로 변환
    port = int( os.getenv('port') ), 
    user = os.getenv('user'), 
    password = os.getenv('pwd'), 
    db = os.getenv('db_name')
)

In [31]:
# Cursor 생성 (select의 결과값을 Dict 형태로 받아온다.)
cursor = _db.cursor( pymysql.cursors.DictCursor )

In [32]:
# 새로운 환경에서 작업이 시작이 될때 모든 테이블의 구조를 다음과 같은 쿼리문을 이용하여 생성 

# 테이블 생성 -> 기존의 테이블이 존재한다면 테이블 생성x -> 없으면 생성
create_table = """
    CREATE TABLE IF NOT EXISTS
    `user_info` 
    (
        `id` VARCHAR(32) PRIMARY KEY, 
        `password` VARCHAR(32) NOT NULL, 
        `name` VARCHAR(32), 
        `age` INT
    )
"""
# DDL 구문은 생성 -> cursor에 질의를 보낸다 -> 테이블을 생성하는 과정은 확인 절차 없이 DB server에 바로 등록 
# cursor를 통해서 sql 쿼리문 실행 -> execute() 함수를 이용
cursor.execute(create_table)

0

In [33]:
# 테이블에 데이터를 대입 -> insert 구문 
signup_query = """
    INSERT INTO `user_info` 
    VALUES ("test", "1234", "kim", 30)
"""

cursor.execute(signup_query)

IntegrityError: (1062, "Duplicate entry 'test' for key 'user_info.PRIMARY'")

In [ ]:
user_list_query = """
    SELECT * FROM `user_info`
"""

cursor.execute(user_list_query)


1

In [ ]:
# 결과 값을 불러오는 함수 -> fetchall()
cursor.fetchall()

[{'id': 'test', 'password': '1234', 'name': 'kim', 'age': 30}]

In [ ]:
# 결과를 DB와의 동기화(확인 후 확정)
# _db에서 확정 작업(commit())
_db.commit()

In [ ]:
# query 문에 들어오는 데이터가 매번 바뀔때마다 query문을 작성하는건 불필요한 작업 
signup_query = """
    INSERT INTO `user_info`
    VALUES (%s, %s, %s, %s)
"""
# %s는 들어오는 데이터들의 위치를 지정 
input_id = input("아이디를 입력하시오")
input_pass = input("비밀번호를 입력하시오")
input_name = input('이름을 입력하시오')
input_age = input('나이를 입력하시오')

# execute(query, [datas]) 함수에 리스트의 각각의 원소들이 %s 위치에 순서대로 대입 
cursor.execute(signup_query, [ input_id, input_pass, input_name, input_age ])

1

In [ ]:
cursor.execute(user_list_query)

2

In [ ]:
cursor.fetchall()

[{'id': 'test', 'password': '1234', 'name': 'kim', 'age': 30},
 {'id': 'test2', 'password': 'park', 'name': 'park', 'age': 20}]

In [ ]:
_db.close()

#### DB 연동 class 선언 
- 생성자 함수 
    - 서버의 정보를 입력한다. 
    - class가 생성이 될때마다 다른 DB server의 정보를 담을수 있다. 
    - 2개의 객체를 생성하여 다른 DB server와의 연동 
- 함수 생성 -> 2개 생성 
    - query문 실행 함수 ( 매개변수 : query문(필수 입력), *datas(query에 %s가 존재하지 않으면 필요x) )
        - 서버와의 연결
        - cursor를 생성
        - CUD ( insert, update, delete )
            - query문이 작성 
            - execute() 함수를 이용하여 cursor에 질의를 보낸다 
                - cursor의 테이블에서 데이터의 변화 
        - R ( select )
            - query문이 작성 
            - execute() 함수를 이용하여 cursor에 질의를 보낸다. 
            - cursor에서 데이터를 가져온다 (fetchall())
        - query문의 시작이 select라면 fetchall()함수를 사용하여 데이터를 리턴
    - DataBase에 변화를 주는 함수 
        - DB server에서 commit()를 이용해서 데이터 확정 
        - DB server와의 연결을 종료 (close())
        

In [3]:
# class 선언 
class MyDB:
    # 생성자 함수 (서버의 정보를 받아오기 위해 사용)
    def __init__(self, host, port, user, password, db):
        # 서버의 정보를 인자로 받아와서 객체 내부에서 독립적인 변수에 저장
        self.host = host
        self.port = port
        self.user = user
        self.password = password
        self.db = db

    # 데이터베이스에 변화를 주는 함수 
    def commit(self):
        try:
            # DataBase에 데이터를 확정 (동기화)
            self.db_server.commit()
            # 서버와의 연결을 종료 ( 중요한 부분 )
            self.db_server.close()
            # close() 함수를 사용하더라도 self.db_server의 변수는 사라지지 않는다. 
            # 변수 자체를 제거 
            del self.db_server
        except:
            # 문제가 발생하는 이유는? -> 서버와의 연결이 되지 않은 경우 (self.db_server라는 변수에 데이터가 없거나 아예 존재하지 않는 경우)
            print( "데이터베이스 서버와의 연결이 되어있지 않습니다. sql_query() 함수를 호출하여 서버와의 연결을 해주세요" )
    
    def sql_query(self, query, *datas):
        # query : sql query문이 입력이되는 매개변수 
        # datas : query문에서 사용이 될 데이터의 목록

        # 문제점 : 서버의 재접속으로 commit 전의 데이터가 날아감. 
        # 이미 접속중인 경우 재접속 금지 (2026.04.20 update)
        # 해결 방법 self.db_server에 데이터가 존재한다면? -> 서버의 접속중이다. 
        try:
            self.db_server
            # 변수가 존재하지 않으면 NameError 발생 
            print('접속된 서버가 존재함')
        except:
            # 예외 상황이 발생하면 서버와 연결(self.db_server 라는 변수가 없을 시 실행)
            # DB_server와의 연결 
            self.db_server = pymysql.connect(
                host = self.host, 
                port = self.port, 
                user = self.user, 
                password = self.password, 
                db = self.db
            )
        # cursor 생성 
        cursor = self.db_server.cursor(pymysql.cursors.DictCursor)

        # self.변수명 // 변수명의 차이는?
            # self.변수 -> 독립적으로 객체 안에 저장이 되는 변수 ( 함수 호출 후에도 데이터가 존재 )
            # 변수 -> 함수 호출시 생성이 되고 함수가 종료가 되면 휘발성으로 사라짐
        try:
            # CUD의 경우에는 execute() 쿼리문을 커서에 질의 보낸다. R (select도 여기까지는 공통의 작업)
            # execute( query, () ) -> 호출 가능 
            # execute( query, (1,2,3) ) -> 호출 가능 
            cursor.execute(query, datas)
            # query가 select문이라면? 
            # select * from table // SELECT * FROM table, """   select * from table   """ -> 두가지의 경우 모두 참 
            # 좌측 공백을 제거 , 소문자를 통일 , 시작값이 select와 같은가 startwith()
            if query.lstrip().lower().startswith('select'):
                # 결과값을 받아온다. 
                result = cursor.fetchall()
            else:
                result = "Query OK!"
            return result
        except Exception as e:
            print('query문 execute중 에러')
            print(e)

In [4]:
# class 생성 (서버의 정보를 등록)
db1 = MyDB(
    host = os.getenv('host'), 
    port = int(os.getenv('port')), 
    user = os.getenv('user'),
    password = os.getenv('pwd'), 
    db = os.getenv('db_name')
)

In [6]:
# 서버와의 연결 
db1.sql_query('select * from `user_info`')

접속된 서버가 존재함


[{'id': 'test', 'password': '1234', 'name': 'kim', 'age': 30},
 {'id': 'test3', 'password': '0123', 'name': 'lee', 'age': 40}]

In [46]:
# 서버와의 연결을 종료
db1.commit()

In [39]:
load_dotenv()

True

In [72]:
os.getenv('port')

'3306'

In [48]:
# 외부 서버와의 연결 테스트 
db2 = MyDB(
    host = os.getenv('host2'), 
    port = int(os.getenv('port2')), 
    user = os.getenv('user2'), 
    password= os.getenv('pwd2'),
    db = os.getenv('db_name2')
)

In [ ]:
db2.sql_query('select * from `AAPL`')

In [52]:
db2.commit()

In [62]:
# select 확인이 끝났으니 insert update delete 확인 
insert_query = """
    INSERT INTO `user_info`
    VALUES (%s, %s, %s, %s)
"""
select_query = """
    SELECT * FROM `user_info`
"""

data_list = ['test3', '0000', 'lee', 40]
db1.sql_query(insert_query, *data_list)

'Query OK!'

In [63]:
db1.sql_query(select_query)
# 어라? 데이터 대입했는데 데이터가 없다?? -> 이유가 멀까? ->
# sql_query() -> 서버에 접속() -> insert 후 commit()하지 않고 다시 서버에 접속을 하면 그 전의 데이터를 사라짐
# 수정 사항 -> 이미 서버에 접속이 되어있다면 재접속을 하지 않는다. 

접속된 서버가 존재함


[{'id': 'test', 'password': '1234', 'name': 'kim', 'age': 30},
 {'id': 'test3', 'password': '0000', 'name': 'lee', 'age': 40}]

In [67]:
update_query = """
    UPDATE `user_info` SET `password` = %s
    WHERE `id` = %s
"""

data_list = ['0123', 'test3']
# query에 데이터를 대입 할때 %s의 개수와 data들의 개수는 같아야한다. -> 다른 경우는 에러가 발생
db1.sql_query(update_query, *data_list)

접속된 서버가 존재함
query문 execute중 에러
not all arguments converted during string formatting


In [65]:
db1.sql_query(select_query)

접속된 서버가 존재함


[{'id': 'test', 'password': '1234', 'name': 'kim', 'age': 30},
 {'id': 'test3', 'password': '0123', 'name': 'lee', 'age': 40}]

In [69]:
db1.commit()

In [7]:
delete_query = """
    DELETE FROM `user_info`
    WHERE `id` = %s
"""

data_list = ['test']

db1.sql_query(delete_query, *data_list)

# Lock이 걸려서 문제 
    # 트랜젝션 락  -> commit이나 rollback의 문제 
    # 인덱스를 사용 안 해서 문제가 발생  -> primary key가 존재하지 않은 경우 

접속된 서버가 존재함


'Query OK!'

In [75]:
# 외부의 모듈을 로드 
import db

In [76]:
# 모듈 리로드
import importlib
importlib.reload(db)

<module 'db' from 'c:\\Users\\ekfla\\Documents\\GitHub\\multicam_11\\20260420\\db.py'>

In [77]:
db3 = db.MyDB()

In [78]:
db3.sql_query(select_query)

[{'id': 'test', 'password': '1234', 'name': 'kim', 'age': 30},
 {'id': 'test3', 'password': '0123', 'name': 'lee', 'age': 40}]